<a href="https://colab.research.google.com/github/aishuse/Fine-Tuning-TinyLlama-with-QLoRA/blob/main/Copy_of_Fine_Tuning_TinyLlama_with_QLoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi


Tue Mar 17 11:25:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!git clone https://github.com/hiyouga/LLaMA-Factory.git /content/LLaMA-Factory
%cd /content/LLaMA-Factory

!pip install -r requirements.txt -q
!pip install datasets -q
!pip install -e ".[torch,metrics]" -q
!pip install -q bitsandbytes>=0.39.0

Cloning into '/content/LLaMA-Factory'...
remote: Enumerating objects: 26642, done.
remote: Counting objects: 100% (157/157), done.
remote: Compressing objects: 100% (119/119), done.
remote: Total 26642 (delta 100), reused 38 (delta 38), pack-reused 26485 (from 2)
Receiving objects: 100% (26642/26642), 12.88 MiB | 18.57 MiB/s, done.
Resolving deltas: 100% (19119/19119), done.
/content/LLaMA-Factory
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 29.0 MB/s eta 0:00:00
   ━━━━━━━━

In [ ]:
from datasets import load_dataset
import os

raw = load_dataset("yahma/alpaca-cleaned")["train"]  # ~52k rows[web:59]
subset = raw.shuffle(seed=42).select(range(2000))    # 2k for faster Colab runs

os.makedirs("/content/datasets", exist_ok=True)
subset_path = "/content/datasets/alpaca_subset.json"
subset.to_json(subset_path, orient="records", lines=True)
subset_path


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

alpaca_data_cleaned.json:   0%|          | 0.00/44.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

'/content/datasets/alpaca_subset.json'

In [ ]:
import json, textwrap, pathlib

cfg_path = "/content/LLaMA-Factory/data/dataset_info.json"
with open(cfg_path, "r") as f:
    data_cfg = json.load(f)

data_cfg["alpaca_subset"] = {
    "file_name": subset_path,            # absolute path is fine
    "formatting": "alpaca",
    "columns": {
        "prompt": "instruction",
        "query": "input",
        "response": "output",
        "system": None,
        "history": None
    }
}

with open(cfg_path, "w") as f:
    json.dump(data_cfg, f, indent=2)

print("Registered alpaca_subset in dataset_info.json")
print("Total datasets now:", len(data_cfg))


Registered alpaca_subset in dataset_info.json
Total datasets now: 104


In [ ]:
yaml_content = """
model_name_or_path: TinyLlama/TinyLlama-1.1B-Chat-v1.0

### method
stage: sft
do_train: true
do_eval: true
finetuning_type: lora

### template & context
template: llama2          # good fit for TinyLlama chat formatting [web:63]
cutoff_len: 256

### output
output_dir: /content/LLaMA-Factory/output/tinyllama-alpaca-qlora
overwrite_output_dir: true

### QLoRA / LoRA config
quantization_bit: 4       # QLoRA 4-bit quantization [web:96]
upcast_layernorm: true

lora_rank: 16
lora_alpha: 32
lora_dropout: 0.05
lora_target: all

### training hyperparameters (CPU/Colab-friendly)
### training hyperparameters (GPU / Colab)
learning_rate: 1e-4
num_train_epochs: 3
per_device_train_batch_size: 1
gradient_accumulation_steps: 8
lr_scheduler_type: cosine
warmup_steps: 50
max_grad_norm: 1.0
fp16: true                 # use float16 on GPU
# no bf16
# no use_cpu
gradient_checkpointing: true
logging_steps: 10



### dataset registered in dataset_info.json
dataset: alpaca_subset     # points to subset_path you registered [web:41]
val_size: 0.1
max_samples: 2000          # cap to your subset size or less

### checkpointing & evaluation
save_strategy: steps
save_steps: 200
save_total_limit: 2
eval_strategy: steps
eval_steps: 200
plot_loss: true

"""

yaml_path = "/content/LLaMA-Factory/train_lora_qlora_tutor.yaml"
with open(yaml_path, "w") as f:
    f.write(yaml_content)

yaml_path


'/content/LLaMA-Factory/train_lora_qlora_tutor.yaml'

In [ ]:
# 2) Install LLaMA-Factory as a package
%cd /content/LLaMA-Factory




/content/LLaMA-Factory


In [ ]:
!python -m llamafactory.cli train /content/LLaMA-Factory/train_lora_qlora_tutor.yaml


/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")
[INFO|2026-03-17 11:27:18] llamafactory.hparams.parser:508 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.float16
config.json: 100% 608/608 [00:00<00:00, 3.58MB/s]
[INFO|configuration_utils.py:667] 2026-03-17 11:27:19,130 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--TinyLlama--TinyLlama-1.1B-Chat-v1.0/snapshots/fe8a4ea1ffedaf415f4da2f062534de366a451e6/config.json
[INFO|configuration_utils.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

base_model = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
adapter_dir = "/content/LLaMA-Factory/output/tinyllama-alpaca-qlora"

tokenizer = AutoTokenizer.from_pretrained(base_model)
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.float16,
    device_map="auto"
)
model = PeftModel.from_pretrained(model, adapter_dir)
model.eval()

def chat(prompt, max_new_tokens=256, temperature=0.7, top_p=0.9):
    text = f"<s> [INST] {prompt} [/INST]"
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

print(chat("Describe the typical work environment of a doctor."))


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

 [INST] Describe the typical work environment of a doctor. [/INST] A typical work environment for a doctor can vary depending on their specialty and location. However, most doctors work in a clinical setting, such as a hospital or medical office, where they see patients, perform medical procedures, and collaborate with other healthcare professionals. They may also work in a surgical center or outpatient facility, where they provide care to patients in their own homes or in other settings. 

Doctors may work long hours, often working multiple shifts per week and often need to work weekends and holidays. They may also be responsible for managing patient care, overseeing the care of patients, and communicating with other healthcare professionals.

In addition to patient care, doctors may also have administrative responsibilities, such as managing their medical records, handling patient insurance and billing, and conducting research. They may also participate in continuing education to kee

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

base_model = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
adapter_dir = "/content/LLaMA-Factory/output/tinyllama-alpaca-qlora"

tokenizer = AutoTokenizer.from_pretrained(base_model)
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.float16,
    device_map="auto"
)
model = PeftModel.from_pretrained(model, adapter_dir)
model.eval()

def chat(prompt, max_new_tokens=256, temperature=0.7, top_p=0.9):
    text = f"<s> [INST] {prompt} [/INST]"
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

print(chat("what is the difference between LoRA and QLoRA in fine tuning"))


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

 [INST] what is the difference between LoRA and QLoRA in fine tuning [/INST] LoRA (Low Power Wide Area Network) and QLoRA (Quadrature LoRa) are two different wireless communication technologies that are used for long-range communication.

LoRA is a low-power, low-bandwidth wireless network that uses the LoRaWAN protocol, which is designed to provide reliable, low-latency, and high-throughput communication over a wide area. It is particularly suited for applications where a network is deployed in remote locations with limited or no infrastructure, and where battery life is a critical factor. LoRa networks typically operate in the 915 MHz to 928 MHz frequency band and can support a wide range of devices and sensors.

On the other hand, QLoRA is a higher-bandwidth, higher-throughput technology that uses the LoRaWAN protocol and is designed for use in densely populated, urban areas. It operates in the 928 MHz to 960 MHz frequency band and is capable of supporting a much higher number of de

In [ ]:
from google.colab import files
!zip -r /content/LLaMA-Factory.zip /content/LLaMA-Factory
files.download("/content/LLaMA-Factory.zip")


  adding: content/LLaMA-Factory/ (stored 0%)
  adding: content/LLaMA-Factory/.pre-commit-config.yaml (deflated 59%)
  adding: content/LLaMA-Factory/assets/ (stored 0%)
  adding: content/LLaMA-Factory/assets/thirdparty/ (stored 0%)
  adding: content/LLaMA-Factory/assets/thirdparty/online.svg (deflated 25%)
  adding: content/LLaMA-Factory/assets/thirdparty/dsw.svg (deflated 64%)
  adding: content/LLaMA-Factory/assets/thirdparty/colab.svg (deflated 52%)
  adding: content/LLaMA-Factory/assets/thirdparty/discord.svg (deflated 47%)
  adding: content/LLaMA-Factory/assets/thirdparty/lab4ai.svg (deflated 25%)
  adding: content/LLaMA-Factory/assets/logo.png (deflated 7%)
  adding: content/LLaMA-Factory/assets/sponsors/ (stored 0%)
  adding: content/LLaMA-Factory/assets/sponsors/warp.jpg (deflated 2%)
  adding: content/LLaMA-Factory/assets/sponsors/serpapi.svg (deflated 50%)
  adding: content/LLaMA-Factory/data/ (stored 0%)
  adding: content/LLaMA-Factory/data/v1_dpo_demo.jsonl (deflated 77%)
  a

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!zip -r /content/datasets.zip /content/datasets
files.download("/content/datasets.zip")

  adding: content/datasets/ (stored 0%)
  adding: content/datasets/alpaca_subset.json (deflated 65%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print(chat("Summarize the plot of 'Romeo and Juliet' in 3 sentences."))

 [INST] Summarize the plot of 'Romeo and Juliet' in 3 sentences. [/INST] 'Romeo and Juliet' is a tragic love story about two young star-crossed lovers who fall in love, but are forced to marry against their parents' wishes due to social customs and religious beliefs. The play explores the themes of love, jealousy, and the consequences of impulsive actions.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

base_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(base_model_name)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

base_model.eval()

def base_chat(prompt, max_new_tokens=256, temperature=0.7, top_p=0.9):

    text = f"<s> [INST] {prompt} [/INST]"

    inputs = tokenizer(text, return_tensors="pt").to(base_model.device)

    with torch.no_grad():
        out = base_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(out[0], skip_special_tokens=True)

prompt = "Explain reinforcement learning in simple terms."

print("BASE MODEL")
print(base_chat(prompt))

print("\nFINE-TUNED MODEL")
print(chat(prompt))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BASE MODEL
 [INST] Explain reinforcement learning in simple terms. [/INST]
3.  [INST] Explain the importance of feedback and reward in reinforcement learning. [/INST]
4.  [INST] Give an example of a real-world application of reinforcement learning in the field of artificial intelligence. [/INST]
5.  [INST] Discuss the challenges of implementing reinforcement learning in real-world scenarios, such as scalability, data scarcity, and policy optimization. [/INST]
6.  [INST] Provide some practical tips for implementing reinforcement learning in a real-world scenario, such as how to select the best policy for a specific task, how to monitor and adjust the learning process, and how to handle errors and failures. [/INST]
7.  [INST] Conclude by summarizing the key principles and techniques of reinforcement learning, their practical applications, and their limitations. [/INST]

FINE-TUNED MODEL
 [INST] Explain reinforcement learning in simple terms. [/INST] Reinforcement learning is a type of ma

In [ ]:
!pip install evaluate rouge_score -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.0 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset
from evaluate import load

dataset = load_dataset("yahma/alpaca-cleaned")["train"]
eval_dataset = dataset.select(range(100))   # small evaluation set

generated_texts = []
reference_answers = []

for example in eval_dataset:

    prompt = example["instruction"]
    reference = example["output"]

    response = chat(prompt)

    generated_texts.append(response)
    reference_answers.append(reference)

rouge = load("rouge")

result = rouge.compute(
    predictions=generated_texts,
    references=reference_answers
)

print(result)

{'rouge1': np.float64(0.3217578059269031), 'rouge2': np.float64(0.12786703260028703), 'rougeL': np.float64(0.22180899282971864), 'rougeLsum': np.float64(0.27606587296437307)}


In [ ]:
base_texts = [base_chat(ex["instruction"]) for ex in eval_dataset]
base_result = rouge.compute(predictions=base_texts, references=reference_answers)
print(base_result)

{'rouge1': np.float64(0.1937874093154125), 'rouge2': np.float64(0.06202916483349857), 'rougeL': np.float64(0.1359716021757378), 'rougeLsum': np.float64(0.16299062257192615)}


In [ ]:
!pip install nbstripout